# Dual local/global attention MoE + LoRMA + DoRA for NER
Configuration: width512, 12 layers, 8 query heads, 4 KV heads, FF2048, 8 experts/top2, 2048-token context. Experts retain pretrained FF width1024 each.
Set the exact matching Phase-1 checkpoint tag and tokenizer path.
Original word grouping and BIO pooling are retained; chunk length is2048 and training batch size2. No LSTM or CRF.
The pretrained MoE routes two experts for the entire batch. Padding is excluded from its routing mean. Predictions can consequently depend on batch composition; inference remains batch size1.
Training optimizes BIO cross-entropy plus the sum of existing per-layer balancing losses. Validation loss is BIO cross-entropy only.
Adapters cover Q/K/V/output attention projections. Last two complete blocks, including their routers/experts, are fully tuned by default.
All20 epochs run; the best validation checkpoint is restored. Validation retains the original small split and reconstructed BIO gold.


In [1]:
import os
import json
import random
import shutil
import tempfile

import torch

from huggingface_hub import login, HfApi, hf_hub_download
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
!pip install -q tokenizers 
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 2.8 MB/s eta 0:00:00


In [3]:
!pip install -q bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.0 MB/s eta 0:00:00


In [4]:
import re
import ijson
from collections import Counter

from tokenizers import ByteLevelBPETokenizer

## Configuration


In [5]:
# =========================
# PROJECT CONFIGURATION
# =========================

# Hugging Face
HF_USERNAME = "A-240031"
HF_REPO_NAME = "legal-lm-phase1"
HF_MODEL_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"
# Dataset files
DATASET_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
}

# Checkpointing
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

# Device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


#TOKENIZER CONFIGURATION
VOCAB_SIZE = 32_768
MIN_FREQUENCY = 5
CUSTOM_TOKEN_MIN_FREQUENCY = 100

SPECIAL_TOKENS = [
    "<|pad|>",
    "<|unk|>",
    "<|bos|>",
    "<|eos|>",
]

ARCH_TRAIN_TOKENS = 50_000_000
VAL_EXAMPLES = 500
VAL_PROMPT_WORDS = 512
VAL_TARGET_WORDS = 128
VAL_DOC_FRACTION = 0.01
SEED = 42

ARCH_TRAIN_PATH = "/kaggle/working/architecture_data/arch_train_200M.bin"
ARCH_VAL_PATH = "/kaggle/working/architecture_data/arch_val_200M.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# =========================
# TRAINING CONFIGURATION
# =========================


In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [7]:
BUCKET_NAME = "A-240031/legal-lm-phase1-bucket"
BUCKET_ROOT = f"hf://buckets/{BUCKET_NAME}"

from huggingface_hub import HfFileSystem

fs = HfFileSystem()

In [8]:
def upload_to_bucket(local_path, remote_path):
    remote_path = remote_path.lstrip("/")

    fs.put_file(
        local_path,
        f"{BUCKET_ROOT}/{remote_path}",
    )

    print(f"Uploaded: {remote_path}")

In [9]:
def download_from_bucket(remote_path, local_path):
    remote_path = remote_path.lstrip("/")

    fs.get_file(
        f"{BUCKET_ROOT}/{remote_path}",
        local_path,
    )

    print(f"Downloaded: {remote_path} -> {local_path}")

## Checkpoint related functions

In [10]:
def upload_checkpoint(tmp_dir, tag):

    remote_dir = f"checkpoints/{tag}"

    for filename in os.listdir(tmp_dir):
        local_path = os.path.join(tmp_dir, filename)

        upload_to_bucket(
            local_path,
            f"{remote_dir}/{filename}",
        )

In [11]:
def save_checkpoint(tag, model, optimizer, scheduler=None, scaler=None,
                    tokens_seen=0, global_step=0, epoch=0, extra_state=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        torch.save(model.state_dict(), os.path.join(tmp_dir, "model.pt"))
        torch.save(optimizer.state_dict(), os.path.join(tmp_dir, "optimizer.pt"))

        if scheduler is not None:
            torch.save(scheduler.state_dict(), os.path.join(tmp_dir, "scheduler.pt"))

        if scaler is not None:
            torch.save(scaler.state_dict(), os.path.join(tmp_dir, "scaler.pt"))

        rng_state = {"python": random.getstate(), "torch": torch.get_rng_state()}

        if torch.cuda.is_available():
            rng_state["cuda"] = torch.cuda.get_rng_state_all()

        torch.save(rng_state, os.path.join(tmp_dir, "rng_state.pt"))

        state = {
            "tag": tag,
            "tokens_seen": tokens_seen,
            "global_step": global_step,
            "epoch": epoch,
            "extra_state": extra_state,
        }

        with open(os.path.join(tmp_dir, "training_state.json"), "w") as f:
            json.dump(state, f, indent=2)

        upload_checkpoint(tmp_dir,tag)

def load_checkpoint(tag, model, optimizer, scheduler=None, scaler=None):

    with tempfile.TemporaryDirectory() as tmp_dir:

        files = [
            "model.pt",
            "optimizer.pt",
            "scheduler.pt",
            "scaler.pt",
            "rng_state.pt",
            "training_state.json",
        ]

        checkpoint_dir = os.path.join(
            tmp_dir,
            "checkpoints",
            tag,
        )

        os.makedirs(checkpoint_dir, exist_ok=True)

        for filename in files:
            try:
                download_from_bucket(
                    f"checkpoints/{tag}/{filename}",
                    os.path.join(checkpoint_dir, filename),
                )
            except Exception:
                pass

        model.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "model.pt"),
                map_location=DEVICE,
                weights_only=True,
            )
        )

        optimizer.load_state_dict(
            torch.load(
                os.path.join(checkpoint_dir, "optimizer.pt"),
                map_location=DEVICE,
                weights_only=False,
            )
        )

        scheduler_path = os.path.join(checkpoint_dir, "scheduler.pt")

        if scheduler is not None and os.path.exists(scheduler_path):
            scheduler.load_state_dict(
                torch.load(
                    scheduler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        scaler_path = os.path.join(checkpoint_dir, "scaler.pt")

        if scaler is not None and os.path.exists(scaler_path):
            scaler.load_state_dict(
                torch.load(
                    scaler_path,
                    map_location=DEVICE,
                    weights_only=False,
                )
            )

        rng_state = torch.load(
            os.path.join(checkpoint_dir, "rng_state.pt"),
            map_location="cpu",
            weights_only=False,
        )

        random.setstate(rng_state["python"])
        torch.set_rng_state(rng_state["torch"])

        if torch.cuda.is_available() and "cuda" in rng_state:
            torch.cuda.set_rng_state_all(rng_state["cuda"])

        with open(
            os.path.join(checkpoint_dir, "training_state.json")
        ) as f:
            state = json.load(f)

    return state

## Loading training data

In [12]:
PHASE1_TAG = "dual_attention_100M"
DUAL_TOKENIZER_PATH = "tokenizer_dual_attention/tokenizer.json"
if DUAL_TOKENIZER_PATH.startswith("SET_"):
    raise ValueError("Set DUAL_TOKENIZER_PATH to the matching Phase-1 tokenizer bucket path")


In [13]:
from pathlib import Path
from tokenizers import Tokenizer

LOCAL_DIR = Path("tokenizer_dual_attention")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TOKENIZER = LOCAL_DIR / "tokenizer.json"

# Download exact tokenizer from bucket
download_from_bucket(
    DUAL_TOKENIZER_PATH,
    str(LOCAL_TOKENIZER)
)

# Load tokenizer
tokenizer = Tokenizer.from_file(
    str(LOCAL_TOKENIZER)
)

print("Tokenizer loaded successfully")
print("Vocab size:", tokenizer.get_vocab_size())

(…)/tokenizer_dual_attention/tokenizer.json:   0%|          | 0.00/2.54M [00:00<?, ?B/s]

Downloaded: tokenizer_dual_attention/tokenizer.json -> tokenizer_dual_attention/tokenizer.json
Tokenizer loaded successfully
Vocab size: 33887


In [14]:
import json
from huggingface_hub import hf_hub_download

HF_DATASET_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# --------------------------------------------------
# Download
# --------------------------------------------------

fold1_text_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold1-text.json",
    repo_type="dataset"
)

fold1_gold_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold1-gold.json",
    repo_type="dataset"
)

fold2_text_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold2-text.json",
    repo_type="dataset"
)

fold2_gold_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/fold2-gold.json",
    repo_type="dataset"
)

labels_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Train/labels.txt",
    repo_type="dataset"
)

# --------------------------------------------------
# Load JSON
# --------------------------------------------------

with open(fold1_text_path, "r", encoding="utf-8") as f:
    fold1_text = json.load(f)

with open(fold1_gold_path, "r", encoding="utf-8") as f:
    fold1_gold = json.load(f)

with open(fold2_text_path, "r", encoding="utf-8") as f:
    fold2_text = json.load(f)

with open(fold2_gold_path, "r", encoding="utf-8") as f:
    fold2_gold = json.load(f)

# --------------------------------------------------
# Load labels
# --------------------------------------------------

with open(labels_path, "r", encoding="utf-8") as f:
    labels = [line.strip() for line in f if line.strip()]

print("Fold 1 documents:", len(fold1_text))
print("Fold 1 annotations:", len(fold1_gold))

print("Fold 2 documents:", len(fold2_text))
print("Fold 2 annotations:", len(fold2_gold))

print("Labels:")
print(labels)

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Fold 1 documents: 35
Fold 1 annotations: 35
Fold 2 documents: 35
Fold 2 annotations: 35
Labels:
['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [15]:
VOCAB_SIZE = tokenizer.get_vocab_size()

BATCH_SIZE = 4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

ARCH_TRAIN_TOKENS = 100_000_000
CHECKPOINT_INTERVAL_TOKENS = 1_000_000

In [16]:
VOCAB_SIZE

33887

## Model - A1 

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, head_dim, 2).float() / head_dim
            )
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos(),
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin(),
            persistent=False,
        )

    def forward(self, q, k):

        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU Expert
# ============================================================

class SwiGLU(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.up_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.down_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(self.gate_proj(x))
            * self.up_proj(x)
        )


# ============================================================
# Mixture of Experts
# ============================================================
class MoE(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        # Router chooses experts BEFORE any expert is evaluated
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False,
        )

        expert_d_ff = d_ff // top_k

        self.experts = nn.ModuleList([
            SwiGLU(
                d_model,
                expert_d_ff,
            )
            for _ in range(n_experts)
        ])

    def forward(self, x, valid_mask=None):

        # x: [B, T, D]

        # ------------------------------------------------
        # Batch representation
        # ------------------------------------------------

        batch_repr = (x.mean(dim=(0, 1)) if valid_mask is None else
            (x * valid_mask[..., None]).sum(dim=(0, 1)) / valid_mask.sum().clamp_min(1))  # [D]

        # ------------------------------------------------
        # ROUTE FIRST
        # ------------------------------------------------

        router_logits = self.router(
            batch_repr
        )  # [E]

        router_probs = F.softmax(
            router_logits,
            dim=-1,
        )

        topk_probs, topk_indices = torch.topk(
            router_probs,
            self.top_k,
            dim=-1,
        )

        # Normalize selected weights
        topk_probs = topk_probs / (
            topk_probs.sum() + 1e-9
        )

        # ------------------------------------------------
        # ONLY NOW evaluate selected experts
        # ------------------------------------------------

        output = 0.0

        for i in range(self.top_k):

            expert_idx = topk_indices[i].item()
            weight = topk_probs[i]

            expert_output = self.experts[
                expert_idx
            ](x)

            output = (
                output
                + weight * expert_output
            )

        # ------------------------------------------------
        # Batch routing loss
        # ------------------------------------------------

        # Router probability distribution for this batch.
        # Used to discourage permanent collapse to
        # a small subset of experts.
        balance_loss = (
            self.balance_alpha
            * self.n_experts
            * torch.sum(
                router_probs
                * router_probs
            )
        )

        return output, balance_loss

# ============================================================
# GQA + SDPA
# ============================================================
class GQAAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        local_window=64,
        global_stride=128,
        max_seq_len=2048,
    ):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.local_window = local_window
        self.global_stride = global_stride
        self.max_seq_len = max_seq_len

        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def _build_sparse_mask(self, T, device):
        """
        Allowed attention for query position i:

        1. Previous local_window positions
        2. All previous global positions
           (every global_stride tokens)
        3. Itself

        Future positions are never allowed.
        """

        positions = torch.arange(
            T,
            device=device,
        )

        # i = query position
        # j = key position
        i = positions[:, None]
        j = positions[None, :]

        # Causal condition
        causal = j <= i

        # Local attention
        local = j >= (i - self.local_window + 1)

        # Global landmark positions
        global_pos = (j % self.global_stride) == 0

        # Allowed if local OR global, while remaining causal
        allowed = causal & (local | global_pos)

        # SDPA bool mask:
        # True  = allowed to attend
        # False = masked
        return allowed

    def forward(self, x, valid_mask=None):

        B, T, _ = x.shape

        # --------------------------------------------------
        # Q, K, V projections
        # --------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # --------------------------------------------------
        # Reshape into heads
        # --------------------------------------------------

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        # --------------------------------------------------
        # RoPE
        # --------------------------------------------------

        q, k = self.rope(q, k)

        # --------------------------------------------------
        # GQA: expand K/V heads
        # --------------------------------------------------

        repeat_factor = (
            self.n_heads // self.n_kv_heads
        )

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # --------------------------------------------------
        # Local + Global sparse attention
        # --------------------------------------------------

        attn_mask = self._build_sparse_mask(
            T,
            x.device,
        )

        if valid_mask is not None:
            attn_mask = attn_mask[None, None] & valid_mask[:, None, None, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            is_causal=False,
        )

        # --------------------------------------------------
        # Merge heads
        # --------------------------------------------------

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block with MoE
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(
            d_model
        )

        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(
            d_model
        )

        self.moe = MoE(
            d_model=d_model,
            d_ff=d_ff,
            n_experts=n_experts,
            top_k=top_k,
            balance_alpha=balance_alpha,
        )

    def forward(self, x, valid_mask=None):

        # Attention
        x = x + self.attention(
            self.norm1(x), valid_mask=valid_mask
        )

        # MoE FFN
        moe_output, balance_loss = self.moe(
            self.norm2(x), valid_mask=valid_mask
        )

        x = x + moe_output

        return x, balance_loss


# ============================================================
# Decoder-only MoE Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,

        # MoE
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                n_experts=n_experts,
                top_k=top_k,
                balance_alpha=balance_alpha,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(
            d_model
        )

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.Embedding):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.RMSNorm):

            nn.init.ones_(
                module.weight
            )

    def forward(
        self,
        input_ids,
        return_aux_loss=False,
    ):

        x = self.token_embedding(
            input_ids
        )

        total_balance_loss = x.new_zeros(
            ()
        )

        for block in self.blocks:

            x, balance_loss = block(x)

            total_balance_loss = (
                total_balance_loss
                + balance_loss
            )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        if return_aux_loss:

            return (
                logits,
                total_balance_loss,
            )

        return logits
class LormaDoRALinear(nn.Module):
    def __init__(
        self,
        original_linear,
        r=16,
        alpha=32,
        dropout=0.05,
        do_pi=True,
        eps=1e-8,
    ):
        super().__init__()

        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.do_pi = do_pi
        self.eps = eps

        self.dropout = nn.Dropout(dropout)

        # Preserve the original pretrained linear layer.
        self.original_linear = original_linear

        # Freeze pretrained weights and bias.
        for param in self.original_linear.parameters():
            param.requires_grad = False

        device = original_linear.weight.device
        dtype = original_linear.weight.dtype

        # LoRMA pre-transformation:
        # B: [out_features, r]
        # A: [r, out_features]
        self.B = nn.Parameter(
            torch.empty(
                self.out_features, r,
                device=device, dtype=dtype,
            )
        )
        self.A = nn.Parameter(
            torch.empty(
                r, self.out_features,
                device=device, dtype=dtype,
            )
        )

        # Initialize the transformation near identity.
        # BA = I when r >= out_features and the initialization
        # is constructed as below; otherwise it is low rank.
        nn.init.zeros_(self.B)
        nn.init.zeros_(self.A)

        with torch.no_grad():
            # This construction makes B @ A = I for any r >= 1:
            # B[:, 0] = 1, A[0, :] = 1 does NOT produce identity.
            # Instead, use a rectangular identity factorization
            # when rank is sufficient.
            k = min(r, self.out_features)
            self.B[:k, :k] = torch.eye(
                k, device=device, dtype=dtype
            )
            self.A[:k, :k] = torch.eye(
                k, device=device, dtype=dtype
            )

        # DoRA-style row magnitudes, initialized from pretrained W.
        initial_magnitude = torch.linalg.vector_norm(
            original_linear.weight.detach().float(),
            ord=2,
            dim=1,
        ).to(device=device, dtype=dtype)

        self.magnitude = nn.Parameter(initial_magnitude)

    def _shift(self, matrix):
        """
        Row-dependent cyclic column shift.
        Matches the shift convention used in our LoRMA implementation.
        """
        n_rows, n_cols = matrix.shape

        row_indices = torch.arange(
            n_rows, device=matrix.device
        ).unsqueeze(1)

        col_indices = torch.arange(
            n_cols, device=matrix.device
        ).unsqueeze(0)

        indices = (col_indices - row_indices) % n_cols

        return torch.gather(matrix, 1, indices)

    def effective_weight(self):
        W = self.original_linear.weight
    
        P_delta = self.B @ self.A
    
        if self.do_pi:
            P_delta = self._shift(P_delta)
    
        identity = torch.eye(
            self.out_features,
            device=W.device,
            dtype=W.dtype,
        )
    
        P = identity + self.scaling * P_delta
        transformed_W = P @ W
    
        row_norms = torch.linalg.vector_norm(
            transformed_W.float(),
            ord=2,
            dim=1,
            keepdim=True,
        ).to(dtype=transformed_W.dtype)
    
        normalized_W = transformed_W / row_norms.clamp_min(self.eps)
    
        return self.magnitude.unsqueeze(1) * normalized_W

    def forward(self, x):
        W_eff = self.effective_weight()

        return F.linear(
            self.dropout(x),
            W_eff,
            self.original_linear.bias,
        )


class EfficientDecoderLMForNER(EfficientDecoderLM):

    def __init__(
        self,
        vocab_size,
        num_labels=25,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__(
            vocab_size=vocab_size,
            d_model=d_model,
            n_heads=n_heads,
            n_kv_heads=n_kv_heads,
            n_layers=n_layers,
            d_ff=d_ff,
            max_seq_len=max_seq_len,
            n_experts=n_experts, top_k=top_k, balance_alpha=balance_alpha,
        )

        self.ner_head = nn.Linear(d_model, num_labels)

    def forward(self, input_ids, word_groups, return_aux_loss=False):
    # input_ids: [B, T]
        B = input_ids.size(0)
    
        x = self.token_embedding(input_ids)
    
        lengths = torch.tensor(
            [max(i for group in groups for i in group) + 1 for groups in word_groups],
            device=input_ids.device,
        )
        mask = torch.arange(input_ids.size(1), device=input_ids.device)[None] < lengths[:, None]
        balance_total = x.new_zeros(())
        for block in self.blocks:
            x, balance = block(x, valid_mask=mask)
            balance_total = balance_total + balance
    
        x = self.final_norm(x)
    
        batch_word_hidden = []
        max_words = 0
    
        for b in range(B):
    
            word_hidden = []
    
            for group in word_groups[b]:
                h = x[b, group].mean(dim=0)
                word_hidden.append(h)
    
            word_hidden = torch.stack(word_hidden)
    
            batch_word_hidden.append(word_hidden)
    
            max_words = max(
                max_words,
                word_hidden.size(0)
            )
    
        # [B, max_words, D]
        padded_word_hidden = x.new_zeros(
            B,
            max_words,
            x.size(-1)
        )
    
        for b, word_hidden in enumerate(batch_word_hidden):
    
            n_words = word_hidden.size(0)
    
            padded_word_hidden[
                b, :n_words
            ] = word_hidden
    
        # [B, max_words, num_labels]
        logits = self.ner_head(padded_word_hidden)
    
        return (logits, balance_total) if return_aux_loss else logits

In [18]:
d_model = 512
n_layers = 12
n_heads = 8
n_kv_heads = 4
d_ff = 2048
seq_len = 2048
n_experts = 8
top_k = 2
MODEL_KWARGS = dict(vocab_size=VOCAB_SIZE, d_model=d_model, n_layers=n_layers,
    n_heads=n_heads, n_kv_heads=n_kv_heads, d_ff=d_ff, max_seq_len=seq_len,
    n_experts=n_experts, top_k=top_k)


In [19]:
teacher_base = EfficientDecoderLM(**MODEL_KWARGS).to(DEVICE)
print(f"Total parameters: {sum(p.numel() for p in teacher_base.parameters())/1e6:.2f}M")


Total parameters: 177.84M


In [20]:
if PHASE1_TAG == "SET_DUAL_ATTENTION_CHECKPOINT_TAG":
    raise ValueError("Set PHASE1_TAG to the exact dual-attention MoE Phase-1 checkpoint")
teacher_optimizer_dummy = torch.optim.AdamW(teacher_base.parameters(), lr=LEARNING_RATE)
teacher_state = load_checkpoint(PHASE1_TAG, teacher_base, teacher_optimizer_dummy)
print("Loaded dual-attention MoE Phase-1 checkpoint")


(…)checkpoints/dual_attention_100M/model.pt:   0%|          | 0.00/712M [00:00<?, ?B/s]

Downloaded: checkpoints/dual_attention_100M/model.pt -> /tmp/tmp7kb_eiiy/checkpoints/dual_attention_100M/model.pt


(…)kpoints/dual_attention_100M/optimizer.pt:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Downloaded: checkpoints/dual_attention_100M/optimizer.pt -> /tmp/tmp7kb_eiiy/checkpoints/dual_attention_100M/optimizer.pt


(…)kpoints/dual_attention_100M/rng_state.pt:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Downloaded: checkpoints/dual_attention_100M/rng_state.pt -> /tmp/tmp7kb_eiiy/checkpoints/dual_attention_100M/rng_state.pt


(…)/dual_attention_100M/training_state.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

Downloaded: checkpoints/dual_attention_100M/training_state.json -> /tmp/tmp7kb_eiiy/checkpoints/dual_attention_100M/training_state.json
Loaded dual-attention MoE Phase-1 checkpoint


In [21]:
del teacher_optimizer_dummy
del teacher_state
import gc
gc.collect()

53

In [22]:
labels

['APP',
 'RESP',
 'A.COUNSEL',
 'R.COUNSEL',
 'JUDGE',
 'WIT',
 'AUTH',
 'COURT',
 'STAT',
 'PREC',
 'DATE',
 'CASENO']

In [23]:
teacher = EfficientDecoderLMForNER(num_labels=25, **MODEL_KWARGS).to(DEVICE)


In [24]:
missing, unexpected = teacher.load_state_dict(teacher_base.state_dict(), strict=False)
assert set(missing) == {"ner_head.weight", "ner_head.bias"}, missing
assert not unexpected, unexpected
print("Transferred complete pretrained backbone")


Transferred complete pretrained backbone


In [25]:
for block in teacher.blocks:
    attn = block.attention
    attn.q_proj = LormaDoRALinear(
    attn.q_proj,
    r=16,
    alpha=32,
    dropout=0.05,
    do_pi=True,
)

    attn.k_proj = LormaDoRALinear(
        attn.k_proj,
        r=16,
        alpha=32,
        dropout=0.05,
        do_pi=True,
    )
    
    attn.v_proj = LormaDoRALinear(
        attn.v_proj,
        r=16,
        alpha=32,
        dropout=0.05,
        do_pi=True,
    )
    
    attn.out_proj = LormaDoRALinear(
        attn.out_proj,
        r=16,
        alpha=32,
        dropout=0.05,
        do_pi=True,
    )

In [26]:
for p in teacher.parameters():
    p.requires_grad = False

In [27]:
for block in teacher.blocks:
    attn = block.attention

    for proj in [
        attn.q_proj,
        attn.k_proj,
        attn.v_proj,
        attn.out_proj,
    ]:
        proj.A.requires_grad = True
        proj.B.requires_grad = True
        proj.magnitude.requires_grad = True
# Unfreeze both hierarchical NER heads
for p in teacher.ner_head.parameters():
    p.requires_grad = True

In [28]:
# Match the prior hybrid setup: adapters plus full tuning of final two blocks.
# Set to 0 for pure PEFT (adapters + classifier only).
FULL_TUNE_LAST_BLOCKS = 2
if FULL_TUNE_LAST_BLOCKS:
    for block in teacher.blocks[-FULL_TUNE_LAST_BLOCKS:]:
        for parameter in block.parameters():
            parameter.requires_grad = True


In [29]:
# Entity labels from labels.txt
print(labels)

BIO_LABELS = ["O"]

for entity in labels:
    BIO_LABELS.append(f"B-{entity}")
    BIO_LABELS.append(f"I-{entity}")

label2id = {
    label: i
    for i, label in enumerate(BIO_LABELS)
}

id2label = {
    i: label
    for label, i in label2id.items()
}

print("Number of BIO labels:", len(BIO_LABELS))
print(label2id)

['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Number of BIO labels: 25
{'O': 0, 'B-APP': 1, 'I-APP': 2, 'B-RESP': 3, 'I-RESP': 4, 'B-A.COUNSEL': 5, 'I-A.COUNSEL': 6, 'B-R.COUNSEL': 7, 'I-R.COUNSEL': 8, 'B-JUDGE': 9, 'I-JUDGE': 10, 'B-WIT': 11, 'I-WIT': 12, 'B-AUTH': 13, 'I-AUTH': 14, 'B-COURT': 15, 'I-COURT': 16, 'B-STAT': 17, 'I-STAT': 18, 'B-PREC': 19, 'I-PREC': 20, 'B-DATE': 21, 'I-DATE': 22, 'B-CASENO': 23, 'I-CASENO': 24}


In [30]:
def prepare_lner_data(text_data, gold_data, tokenizer):

    processed_data = {}

    for doc_id, text in text_data.items():

       

        # --------------------------------------------------
        # 1. Tokenize
        # --------------------------------------------------

        encoding = tokenizer.encode(text)

        tokens = encoding.tokens
        input_ids = encoding.ids
        offsets = encoding.offsets

        # --------------------------------------------------
        # 2. Group tokens
        #
        # Rule:
        #   Ġ  -> new word
        #   no Ġ -> continue previous word
        # --------------------------------------------------

        word_groups = []
        current_group = []

        for i, token in enumerate(tokens):

            start, end = offsets[i]

            if start == end:
                continue

            if current_group and token.startswith("Ġ"):
                word_groups.append(current_group)
                current_group = []

            current_group.append(i)

        if current_group:
            word_groups.append(current_group)

        # --------------------------------------------------
        # 3. Correct character spans
        #
        # Raw tokenizer offsets include the whitespace
        # represented by Ġ.
        # Remove that whitespace from the beginning.
        # --------------------------------------------------

        word_spans = []

        for group in word_groups:

            start = offsets[group[0]][0]
            end = offsets[group[-1]][1]

            while start < end and text[start].isspace():
                start += 1

            word_spans.append((start, end))

        # --------------------------------------------------
        # 4. Initialize labels
        # --------------------------------------------------

        word_labels = ["O"] * len(word_groups)

        # --------------------------------------------------
        # 5. Align gold entities
        # --------------------------------------------------
        if gold_data is not None:
            gold_entities = gold_data[doc_id]
            for entity_start, entity_end, entity_type, entity_text in gold_entities:
    
                overlapping_words = []
    
                for word_idx, (word_start, word_end) in enumerate(word_spans):
    
                    if word_start < entity_end and word_end > entity_start:
                        overlapping_words.append(word_idx)
    
                if not overlapping_words:
                    continue
    
                word_labels[overlapping_words[0]] = f"B-{entity_type}"
    
                for word_idx in overlapping_words[1:]:
                    word_labels[word_idx] = f"I-{entity_type}"

        # --------------------------------------------------
        # 6. Save
        # --------------------------------------------------

        processed_data[doc_id] = {
            "input_ids": input_ids,
            "offsets": offsets,
            "word_groups": word_groups,
            "word_spans": word_spans,
            "word_labels": word_labels,
        }

    return processed_data

In [31]:
fold1_processed = prepare_lner_data(
    fold1_text,
    fold1_gold,
    tokenizer
)


In [32]:
fold2_processed = prepare_lner_data(
    fold2_text,
    fold2_gold,
    tokenizer
)

In [33]:
MAX_SEQ_LEN = 2048

def make_chunks(example, max_seq_len=2048):
    """
    Split one processed document into chunks at word boundaries.

    Returns list of:
        {
            "input_ids": [...],
            "word_groups": [...],
            "word_labels": [...]
        }
    """

    input_ids = example["input_ids"]
    word_groups = example["word_groups"]
    word_labels = example["word_labels"]
    word_spans = example["word_spans"]
    chunks = []

    current_words = []
    current_start = None
    current_end = None

    for word_idx, group in enumerate(word_groups):

        word_start = group[0]
        word_end = group[-1] + 1

        if current_start is None:
            current_start = word_start
            current_end = word_end
            current_words = [word_idx]
            continue

        # Would adding this word exceed 2048?
        new_end = word_end

        if new_end - current_start > max_seq_len:

            # Save current chunk
            chunk_input_ids = input_ids[current_start:current_end]

            chunk_groups = [
                [
                    token_idx - current_start
                    for token_idx in word_groups[w]
                ]
                for w in current_words
            ]

            chunk_labels = [
                word_labels[w]
                for w in current_words
            ]
            chunk_spans = [
                word_spans[w]
                for w in current_words
            ]
            chunks.append({
                "input_ids": chunk_input_ids,
                "word_groups": chunk_groups,
                "word_labels": chunk_labels,
                "word_spans" : chunk_spans
            })

            # Start new chunk
            current_start = word_start
            current_end = word_end
            current_words = [word_idx]

        else:
            current_end = word_end
            current_words.append(word_idx)

    # Last chunk
    if current_words:

        chunk_input_ids = input_ids[current_start:current_end]

        chunk_groups = [
            [
                token_idx - current_start
                for token_idx in word_groups[w]
            ]
            for w in current_words
        ]

        chunk_labels = [
            word_labels[w]
            for w in current_words
        ]
        chunk_spans = [
            word_spans[w]
            for w in current_words
        ]
        chunks.append({
            "input_ids": chunk_input_ids,
            "word_groups": chunk_groups,
            "word_labels": chunk_labels,
            "word_spans" : chunk_spans
        })

    return chunks

In [34]:
import random

SEED = 42
TRAIN_RATIO = 0.99
MAX_SEQ_LEN = 2048

# --------------------------------------------------
# 1. Combine Fold 1 + Fold 2
# --------------------------------------------------

all_processed_data = {}

all_processed_data.update(fold1_processed)
all_processed_data.update(fold2_processed)

doc_ids = list(all_processed_data.keys())

print("Total documents:", len(doc_ids))

# --------------------------------------------------
# 2. Shuffle documents
# --------------------------------------------------

rng = random.Random(SEED)
rng.shuffle(doc_ids)

# --------------------------------------------------
# 3. 99/1 document-level split
# --------------------------------------------------

split_idx = int(TRAIN_RATIO * len(doc_ids))

train_doc_ids = doc_ids[:split_idx]
val_doc_ids = doc_ids[split_idx:]

print("Train documents:", len(train_doc_ids))
print("Validation documents:", len(val_doc_ids))

print(
    "Train ratio:",
    len(train_doc_ids) / len(doc_ids)
)

print(
    "Validation ratio:",
    len(val_doc_ids) / len(doc_ids)
)

# --------------------------------------------------
# 4. Chunk training documents
# --------------------------------------------------

train_chunks = []

for doc_id in train_doc_ids:

    chunks = make_chunks(
        all_processed_data[doc_id],
        MAX_SEQ_LEN
    )

    train_chunks.extend(chunks)

# --------------------------------------------------
# 5. Chunk validation documents
# --------------------------------------------------

val_chunks = []

for doc_id in val_doc_ids:

    chunks = make_chunks(
        all_processed_data[doc_id],
        MAX_SEQ_LEN
    )

    val_chunks.extend(chunks)

# --------------------------------------------------
# 6. Check
# --------------------------------------------------

print("Train chunks:", len(train_chunks))
print("Validation chunks:", len(val_chunks))

assert len(train_chunks) > 0
assert len(val_chunks) > 0

assert all(
    len(chunk["input_ids"]) <= MAX_SEQ_LEN
    for chunk in train_chunks
)

assert all(
    len(chunk["input_ids"]) <= MAX_SEQ_LEN
    for chunk in val_chunks
)

Total documents: 70
Train documents: 69
Validation documents: 1
Train ratio: 0.9857142857142858
Validation ratio: 0.014285714285714285
Train chunks: 251
Validation chunks: 4


In [35]:
def collate_lner(batch, pad_token_id):

    max_len = max(
        len(x["input_ids"])
        for x in batch
    )

    max_words = max(
        len(x["word_labels"])
        for x in batch
    )

    input_ids = []
    word_groups = []
    word_labels = []

    for x in batch:

        # -------------------------
        # Token padding
        # -------------------------

        ids = x["input_ids"]

        padded = ids + [pad_token_id] * (
            max_len - len(ids)
        )

        input_ids.append(padded)

        # -------------------------
        # Word groups
        # -------------------------

        word_groups.append(
            x["word_groups"]
        )

        # -------------------------
        # Word-label padding
        # -------------------------

        labels = [
            label2id[label]
            for label in x["word_labels"]
        ]

        labels += [-100] * (
            max_words - len(labels)
        )

        word_labels.append(labels)

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long
        ),

        "word_groups": word_groups,

        "word_labels": torch.tensor(
            word_labels,
            dtype=torch.long
        ),
    }

In [36]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_chunks,
    batch_size=2,
    shuffle=True,
    collate_fn=lambda batch: collate_lner(
        batch,
        tokenizer.token_to_id("<|pad|>")
    )
)

val_loader = DataLoader(
    val_chunks,
    batch_size=1,
    shuffle=False,
    collate_fn=lambda batch: collate_lner(
        batch,
        tokenizer.token_to_id("<|pad|>")
    )
)

In [37]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
teacher = teacher.to(device)

NUM_EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
GRAD_ACCUM_STEPS = 4

criterion = nn.CrossEntropyLoss()

trainable_params = [
    p for p in teacher.parameters()
    if p.requires_grad
]

optimizer = AdamW(
    trainable_params,
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in trainable_params)
)

Trainable parameters: 27370009


In [38]:
ENTITY_LABELS = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]


In [39]:
def decode_bio_predictions(
    predictions,
    word_spans,
):
    entities = []

    current_label = None
    current_start = None
    current_end = None

    for label, (start, end) in zip(
        predictions,
        word_spans
    ):

        # A whitespace-only BPE group can become [end, end] after
        # leading whitespace trimming. It cannot form an entity.
        if end <= start:
            if current_label is not None and current_end > current_start:
                entities.append([current_start, current_end, current_label])
            current_label = None
            current_start = None
            current_end = None
            continue

        # O
        if label == "O":

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

                current_label = None
                current_start = None
                current_end = None

            continue

        # B-ENTITY
        if label.startswith("B-"):

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

            current_label = label[2:]
            current_start = start
            current_end = end

            continue

        # I-ENTITY
        if label.startswith("I-"):

            entity_type = label[2:]

            # Continue current entity
            if current_label == entity_type:

                current_end = end

            # Invalid I-ENTITY -> treat as a new entity
            else:

                if current_label is not None:
                    entities.append([
                        current_start,
                        current_end,
                        current_label
                    ])

                current_label = entity_type
                current_start = start
                current_end = end

    # Flush final entity
    if current_label is not None:
        entities.append([
            current_start,
            current_end,
            current_label
        ])

    return [entity for entity in entities if entity[1] > entity[0]]

In [40]:
def bio_to_spans(labels, word_spans):
    """
    Convert BIO word labels + character spans into:
        [start, end, label]
    """

    entities = []

    current_label = None
    current_start = None
    current_end = None

    for label, (start, end) in zip(
        labels,
        word_spans
    ):

        if label == "O":

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

                current_label = None
                current_start = None
                current_end = None

            continue

        if label.startswith("B-"):

            if current_label is not None:
                entities.append([
                    current_start,
                    current_end,
                    current_label
                ])

            current_label = label[2:]
            current_start = start
            current_end = end

        elif label.startswith("I-"):

            entity_type = label[2:]

            if current_label == entity_type:

                current_end = end

            else:
                # Treat invalid I-X as a new entity
                if current_label is not None:
                    entities.append([
                        current_start,
                        current_end,
                        current_label
                    ])

                current_label = entity_type
                current_start = start
                current_end = end

    if current_label is not None:
        entities.append([
            current_start,
            current_end,
            current_label
        ])

    return entities

from tqdm import tqdm
import torch


def generate_val_predictions(
    teacher,
    all_processed_data,
    val_doc_ids,
    id2label,
    max_seq_len=2048,
):
    teacher.eval()

    predicted = {}
    gold = {}

    with torch.no_grad():

        for doc_id in tqdm(
            val_doc_ids,
            desc="Validation prediction"
        ):

            example = all_processed_data[doc_id]

            chunks = make_chunks(
                example,
                max_seq_len
            )

            doc_predictions = []

            for chunk in chunks:

                input_ids = torch.tensor(
                    [chunk["input_ids"]],
                    dtype=torch.long,
                    device=DEVICE
                )

                word_groups = [
                    chunk["word_groups"]
                ]

                logits = teacher(
                    input_ids,
                    word_groups
                )

                predicted_ids = logits.argmax(
                    dim=-1
                )[0]

                predicted_labels = [
                    id2label[idx.item()]
                    for idx in predicted_ids
                ]

                entities = decode_bio_predictions(
                    predicted_labels,
                    chunk["word_spans"]
                )

                doc_predictions.extend(
                    entities
                )

            predicted[doc_id] = doc_predictions

            # Gold spans are obtained directly from the
            # original gold BIO labels + character spans
            gold_labels = example["word_labels"]

            gold[doc_id] = bio_to_spans(
                gold_labels,
                example["word_spans"]
            )

    return predicted, gold

from collections import defaultdict


def evaluate_strict_ner(
    predicted,
    gold,
    labels,
):
    """
    Strict entity-level evaluation.

    Returns:
        Macro-F1
        Macro-Precision
        Macro-Recall
        per-label statistics
    """

    stats = {
        label: {
            "tp": 0,
            "fp": 0,
            "fn": 0,
        }
        for label in labels
    }

    for doc_id in gold:

        gold_entities = gold.get(
            doc_id,
            []
        )

        pred_entities = predicted.get(
            doc_id,
            []
        )

        # Exact entity tuples
        gold_set = set(
            tuple(entity)
            for entity in gold_entities
        )

        pred_set = set(
            tuple(entity)
            for entity in pred_entities
        )

        # True positives
        for entity in pred_set & gold_set:

            label = entity[2]

            if label in stats:
                stats[label]["tp"] += 1

        # False positives
        for entity in pred_set - gold_set:

            label = entity[2]

            if label in stats:
                stats[label]["fp"] += 1

        # False negatives
        for entity in gold_set - pred_set:

            label = entity[2]

            if label in stats:
                stats[label]["fn"] += 1

    # -------------------------
    # Per-label metrics
    # -------------------------

    per_label = {}

    macro_precision = 0.0
    macro_recall = 0.0
    macro_f1 = 0.0

    for label in labels:

        tp = stats[label]["tp"]
        fp = stats[label]["fp"]
        fn = stats[label]["fn"]

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall /
            (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }

        macro_precision += precision
        macro_recall += recall
        macro_f1 += f1

    n = len(labels)

    macro_precision /= n
    macro_recall /= n
    macro_f1 /= n

    return {
        "strict_macro_f1": macro_f1,
        "strict_macro_precision": macro_precision,
        "strict_macro_recall": macro_recall,
        "per_label": per_label,
    }

In [41]:
from collections import Counter
import torch
import torch.nn as nn

label_counts = Counter()

for chunk in train_chunks:
    for label in chunk["word_labels"]:
        label_counts[label2id[label]] += 1

counts = torch.tensor(
    [label_counts[i] for i in range(len(label2id))],
    dtype=torch.float
)

# Inverse square-root frequency to avoid excessively
# large weights for rare labels
weights = 1.0 / torch.sqrt(counts.clamp(min=1))

# Normalize so the average class weight is 1
weights = weights / weights.mean()

# Reduce the weight of O
weights[label2id["O"]] *= 0.25

# Normalize again
weights = weights / weights.mean()

class_weights = weights.to(DEVICE)



print({
    label: round(class_weights[idx].item(), 3)
    for label, idx in label2id.items()
})

{'O': 0.01, 'B-APP': 1.127, 'I-APP': 1.086, 'B-RESP': 1.164, 'I-RESP': 1.035, 'B-A.COUNSEL': 1.622, 'I-A.COUNSEL': 1.675, 'B-R.COUNSEL': 1.67, 'I-R.COUNSEL': 1.791, 'B-JUDGE': 1.453, 'I-JUDGE': 1.344, 'B-WIT': 1.657, 'I-WIT': 1.802, 'B-AUTH': 0.732, 'I-AUTH': 0.744, 'B-COURT': 0.72, 'I-COURT': 0.611, 'B-STAT': 0.65, 'I-STAT': 0.507, 'B-PREC': 0.659, 'I-PREC': 0.278, 'B-DATE': 0.581, 'I-DATE': 0.752, 'B-CASENO': 0.833, 'I-CASENO': 0.496}


In [42]:
def train_teacher(
    teacher,
    train_loader,
    val_loader,
    label2id,
    optimizer,
    trainable_params,
    tag,
    num_epochs=5,
    resume=True,
    resume_tag=None,
    patience=3,
    min_delta=0.001,
):
    device = next(teacher.parameters()).device

    criterion = nn.CrossEntropyLoss(
        ignore_index=-100
    )

    # =========================
    # RESUME
    # =========================

    if patience < 1 or min_delta < 0:
        raise ValueError("patience must be >= 1 and min_delta must be >= 0")
    start_epoch = 0
    best_f1 = float("-inf")
    patience_reference = float("-inf")
    epochs_without_improvement = 0
    history = []
    best_tag = f"{tag}_best"
    have_best = False

    if resume and resume_tag is not None:

        state = load_checkpoint(
            tag=resume_tag,
            model=teacher,
            optimizer=optimizer,
        )

        start_epoch = state["epoch"]
        extra = state.get("extra_state") or {}
        best_f1 = extra.get("best_f1", float("-inf"))
        patience_reference = extra.get("patience_reference", best_f1)
        epochs_without_improvement = extra.get("epochs_without_improvement", 0)
        history = extra.get("history", [])
        best_tag = extra.get("best_tag", best_tag)
        have_best = extra.get("have_best", False)
        # Older checkpoints have no early-stopping state: tracking starts here.

        print(
            f"Resuming from epoch {start_epoch}"
        )

    # =========================
    # TRAINING
    # =========================

    for epoch in range(start_epoch, num_epochs):
        if epochs_without_improvement >= patience:
            print("Resumed checkpoint already reached early-stopping patience")
            break

        teacher.train()

        total_train_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{num_epochs}"
        )

        for step, batch in enumerate(progress):

            input_ids = batch["input_ids"].to(device)

            word_groups = batch["word_groups"]

            targets = batch["word_labels"].to(device)

            optimizer.zero_grad(set_to_none=True)

            # [B, max_words, num_labels]
            logits, auxiliary_loss = teacher(
                input_ids, word_groups, return_aux_loss=True
            )

            # Flatten:
            # logits  -> [B * max_words, num_labels]
            # targets -> [B * max_words]
            loss = criterion(
                logits.view(
                    -1,
                    logits.size(-1)
                ),
                targets.view(-1)
            )

            loss = loss + auxiliary_loss
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                GRAD_CLIP
            )

            optimizer.step()

            total_train_loss += loss.item()

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        avg_train_loss = (
            total_train_loss /
            len(train_loader)
        )

        # =========================
        # VALIDATION
        # =========================

        teacher.eval()

        total_val_loss = 0.0
        
        with torch.no_grad():
        
            for batch in val_loader:
        
                input_ids = batch["input_ids"].to(device)
                word_groups = batch["word_groups"]
                targets = batch["word_labels"].to(device)
        
                logits = teacher(
                    input_ids,
                    word_groups
                )
        
                loss = criterion(
                    logits.view(-1, logits.size(-1)),
                    targets.view(-1)
                )
        
                total_val_loss += loss.item()
        
        avg_val_loss = (
            total_val_loss /
            len(val_loader)
        )
        
        # =========================
        # STRICT VALIDATION F1
        # =========================
        
        val_predicted, val_gold = generate_val_predictions(
            teacher=teacher,
            all_processed_data=all_processed_data,
            val_doc_ids=val_doc_ids,
            id2label=id2label,
            max_seq_len=MAX_SEQ_LEN,
        )
        
        val_metrics = evaluate_strict_ner(
            predicted=val_predicted,
            gold=val_gold,
            labels=ENTITY_LABELS,
        )
        
        val_f1 = val_metrics["strict_macro_f1"]
        # =========================
        # LOGGING
        # =========================

        print(
            f"\nEpoch {epoch + 1}/{num_epochs}"
        )

        print(
            f"Train Loss:    {avg_train_loss:.4f}"
        )

        print(
            f"Val Loss:      {avg_val_loss:.4f}"
        )

        print(
            f"Val Strict F1:  {val_f1:.4f}"
        )

        # Save any strict F1 improvement, even if it is smaller than min_delta.
        improved = val_f1 > best_f1
        if improved:
            best_f1 = val_f1
            have_best = True

        # min_delta controls patience resets, not which checkpoint is best.
        if val_f1 > patience_reference + min_delta:
            patience_reference = val_f1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        history.append({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_strict_macro_f1": val_f1,
        })
        extra = {
            "best_f1": best_f1,
            "patience_reference": patience_reference,
            "epochs_without_improvement": epochs_without_improvement,
            "best_tag": best_tag,
            "have_best": have_best,
            "history": history,
        }
        if improved:
            save_checkpoint(
                tag=best_tag, model=teacher, optimizer=optimizer,
                epoch=epoch + 1, extra_state=extra,
            )
            print(f"New best checkpoint: {best_tag} (F1={best_f1:.4f})")

        # Keep latest state separately so an interrupted run can resume.
        save_checkpoint(
            tag=tag, model=teacher, optimizer=optimizer,
            epoch=epoch + 1, extra_state=extra,
        )
        print(f"Early stopping: {epochs_without_improvement}/{patience}")
        if epochs_without_improvement >= patience:
            print(f"Stopping early. Best validation F1: {best_f1:.4f}")
            break

    if have_best:
        # Mutates the existing teacher, so the existing prediction cells use best weights.
        load_checkpoint(tag=best_tag, model=teacher, optimizer=optimizer)
        print(f"Restored best model from {best_tag} (F1={best_f1:.4f})")
    teacher.eval()
    return teacher, history


In [43]:
teacher, history = train_teacher(
    teacher=teacher,
    train_loader = train_loader,
    val_loader=val_loader,
    label2id=label2id,
    optimizer=optimizer,
    trainable_params=trainable_params,
    tag="dual_attention_moe_lorma_dora",
    num_epochs=20,
    patience=21,
    min_delta=0.0,
)

Validation prediction: 100%|██████████| 1/1 [00:00<00:00,  1.15it/s]



Epoch 1/20
Train Loss:    0.4206
Val Loss:      0.0950
Val Strict F1:  0.2756


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Uploaded: checkpoints/dual_attention_moe_lorma_dora_best/optimizer.pt


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (314 outputs left out)

(…)tention_moe_lorma_dora_best/rng_state.pt:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

Downloaded: checkpoints/dual_attention_moe_lorma_dora_best/rng_state.pt -> /tmp/tmpj8n6z5mb/checkpoints/dual_attention_moe_lorma_dora_best/rng_state.pt


(…)_moe_lorma_dora_best/training_state.json:   0%|          | 0.00/1.03k [00:00<?, ?B/s]

Downloaded: checkpoints/dual_attention_moe_lorma_dora_best/training_state.json -> /tmp/tmpj8n6z5mb/checkpoints/dual_attention_moe_lorma_dora_best/training_state.json
Restored best model from dual_attention_moe_lorma_dora_best (F1=0.3741)


In [44]:
import json
import zipfile
import os
import torch
from tqdm import tqdm


def generate_predictions(
    teacher,
    processed_test,
    id2label,
    tag,
    max_seq_len=2048,
):
    teacher.eval()

    predictions = {}

    with torch.no_grad():

        for doc_id, example in tqdm(
            processed_test.items(),
            desc="Generating predictions"
        ):

            chunks = make_chunks(
                example,
                max_seq_len
            )

            document_entities = []

            for chunk in chunks:

                input_ids = torch.tensor(
                    [chunk["input_ids"]],
                    dtype=torch.long,
                    device=DEVICE
                )

                word_groups = [
                    chunk["word_groups"]
                ]

                logits = teacher(
                    input_ids,
                    word_groups
                )

                predicted_ids = logits.argmax(
                    dim=-1
                )[0]

                predicted_labels = [
                    id2label[idx.item()]
                    for idx in predicted_ids
                ]

                entities = decode_bio_predictions(
                    predicted_labels,
                    chunk["word_spans"]
                )

                document_entities.extend(
                    entities
                )

            predictions[doc_id] = document_entities

    # =========================
    # Save predictions.json
    # =========================

    json_path = f"/kaggle/working/predictions.json"

    with open(json_path, "w") as f:
        json.dump(
            predictions,
            f,
            indent=2
        )

    # =========================
    # Zip as tag.zip
    # =========================

    zip_path = f"/kaggle/working/{tag}.zip"

    with zipfile.ZipFile(
        zip_path,
        "w",
        zipfile.ZIP_DEFLATED
    ) as z:

        z.write(
            json_path,
            arcname="prediction.json"
        )

    print(f"Saved: {zip_path}")

    return zip_path

In [45]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO_ID,
    filename="Phase_2_LNER/Test/fold3-text.json",
    repo_type="dataset",
)

with open(test_file_path, "r") as f:
    fold3_text = json.load(f)

print(f"Loaded {len(fold3_text)} test documents")
processed_test = prepare_lner_data(
    fold3_text,
    None,
    tokenizer
)

print(f"Processed {len(processed_test)} documents")

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Loaded 35 test documents
Processed 35 documents


In [46]:
generate_predictions(
    teacher,
    processed_test,
    id2label,
    "dual_attention_moe_lorma_dora",
    max_seq_len=2048,
)

Generating predictions: 100%|██████████| 35/35 [00:28<00:00,  1.22it/s]

Saved: /kaggle/working/dual_attention_moe_lorma_dora.zip


'/kaggle/working/dual_attention_moe_lorma_dora.zip'